# 00 — Preflight, data gates, and execution order

Validates the local environment and runs or reviews the non-negotiable metadata/media gates.

Every displayed denominator and paper-facing visual is also saved under `outputs/visualization/`. Empty or under-supported analyses remain visible as audit rows; they are never silently removed.

In [ ]:
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "paper1_qc").exists():
            return candidate
    raise FileNotFoundError("Open Jupyter from inside paper1_pipeline_rebuilt.")

ROOT = find_project_root()
CONFIG = ROOT / "config" / "project.yaml"
OUTPUT = ROOT / "outputs"
VIZ_ROOT = OUTPUT / "visualization"
sys.path.insert(0, str(ROOT / "src"))

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def read_stage(relative_without_suffix):
    stem = OUTPUT / relative_without_suffix
    parquet = stem.with_suffix(".parquet")
    csv = stem.with_suffix(".csv")
    if parquet.exists():
        return pd.read_parquet(parquet)
    if csv.exists():
        try:
            return pd.read_csv(csv)
        except pd.errors.EmptyDataError:
            return pd.DataFrame()
    raise FileNotFoundError(f"Missing required stage table: {parquet} or {csv}")

def run_cli(*arguments):
    command = [sys.executable, "-m", "paper1_qc.cli", "--config", str(CONFIG), *arguments]
    print("RUN:", " ".join(map(str, command)))
    subprocess.run(command, cwd=ROOT, check=True)

def save_table(frame, folder, name):
    target = VIZ_ROOT / folder
    target.mkdir(parents=True, exist_ok=True)
    path = target / f"{name}.csv"
    frame.to_csv(path, index=False)
    print("TABLE:", path.relative_to(ROOT), f"({len(frame):,} rows)")
    return path

def save_figure(fig, folder, name):
    target = VIZ_ROOT / folder
    target.mkdir(parents=True, exist_ok=True)
    png = target / f"{name}.png"
    svg = target / f"{name}.svg"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    print("FIGURE:", png.relative_to(ROOT))
    return png, svg

assert CONFIG.exists(), "Copy config/project.example.yaml to config/project.yaml and review it."
print("Project:", ROOT)
print("Config:", CONFIG)
print("Visualization outputs:", VIZ_ROOT)


Set `RUN_PIPELINE_STAGES=True` only after `config/project.yaml` points to the updated data root. The audit is intentionally run before any signal processing.

In [ ]:
RUN_PIPELINE_STAGES = False

if RUN_PIPELINE_STAGES:
    run_cli("audit")
    run_cli("inventory")
else:
    print("Dry review only. Set RUN_PIPELINE_STAGES=True to run audit and inventory.")


In [ ]:
audit_summary = read_stage("00_audit/bamboo_audit_summary")
cross_workbook = read_stage("00_audit/cross_workbook_summary")
metadata_issues = read_stage("00_audit/bamboo_audit_issues")
inventory = read_stage("00_audit/bamboo_media_inventory")

display(audit_summary)
display(cross_workbook)

issue_counts = (
    metadata_issues.groupby(["severity", "issue"], dropna=False)
    .size().rename("n").reset_index().sort_values(["severity", "n"], ascending=[True, False])
)
save_table(issue_counts, "00_preflight", "metadata_issue_counts")
display(issue_counts)

media_summary = pd.DataFrame({
    "recordings_on_disk": [inventory["file_name"].nunique()],
    "physical_files": [len(inventory)],
    "extensions": [", ".join(sorted(inventory["extension"].dropna().astype(str).unique())) if "extension" in inventory else "not available"],
    "probe_failures": [int(inventory.get("probe_status", pd.Series(dtype=str)).astype(str).ne("ok").sum()) if "probe_status" in inventory else np.nan],
})
save_table(media_summary, "00_preflight", "media_inventory_summary")
display(media_summary)


In [ ]:
# Hard-stop ledger: resolve every error before interpreting Q.
blocking = metadata_issues.loc[metadata_issues["severity"].astype(str).str.lower().eq("error")].copy()
save_table(blocking, "00_preflight", "blocking_metadata_issues")
print(f"Blocking metadata rows: {len(blocking):,}")
if len(blocking):
    display(blocking.head(50))
